# ChurnSense — Exploratory Data Analysis

This notebook covers:
1. Dataset overview and schema
2. Target distribution (churn rate)
3. Feature distributions and outlier analysis
4. Correlation analysis
5. Key business insights pre-modelling


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Dark theme for all plots
plt.rcParams.update({
    'figure.facecolor': '#0F1117',
    'axes.facecolor':   '#161B27',
    'axes.edgecolor':   '#232838',
    'text.color':       '#EAEAEA',
    'axes.labelcolor':  '#808898',
    'xtick.color':      '#808898',
    'ytick.color':      '#808898',
    'grid.color':       '#232838',
    'font.family':      'DejaVu Sans',
})

df = pd.read_csv('../data/telco_churn.csv')
print(f'Shape: {df.shape}')
df.head()

## 1. Dataset Overview

In [ ]:
print('Data types:')
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('\nDescriptive stats (numeric):')
df.describe()

## 2. Target Distribution

In [ ]:
churn_counts = df['Churn'].value_counts()
churn_rate   = (df['Churn'] == 'Yes').mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.patch.set_facecolor('#0F1117')

colors = ['#6C63FF', '#FF6B6B']
axes[0].bar(['Retained', 'Churned'], churn_counts.values, color=colors, width=0.5)
axes[0].set_title('Churn Distribution', color='white', fontweight='bold')
axes[0].set_ylabel('Count', color='#808898')
for i, (label, val) in enumerate(zip(['Retained', 'Churned'], churn_counts.values)):
    axes[0].text(i, val + 30, f'{val:,}', ha='center', color='white', fontweight='bold')

axes[1].pie(
    churn_counts.values,
    labels=['Retained', 'Churned'],
    colors=colors,
    autopct='%1.1f%%',
    startangle=90,
    textprops={'color': 'white'},
    wedgeprops={'linewidth': 2, 'edgecolor': '#0F1117'},
)
axes[1].set_title(f'Churn Rate: {churn_rate:.1%}', color='white', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/eda_target_distribution.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()
print(f'Churn rate: {churn_rate:.2%} — dataset is imbalanced ({1/churn_rate:.1f}:1 ratio)')

## 3. Key Feature Distributions

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.patch.set_facecolor('#0F1117')

for i, col in enumerate(numeric_cols):
    ax1, ax2 = axes[0, i], axes[1, i]
    
    # Overall distribution
    ax1.hist(df[col].dropna(), bins=30, color='#6C63FF', alpha=0.8, edgecolor='#0F1117')
    ax1.set_title(f'{col} — Overall', color='white', fontweight='bold')
    ax1.set_xlabel(col, color='#808898')
    ax1.set_ylabel('Count', color='#808898')
    
    # Split by churn
    for label, color in [('No', '#6C63FF'), ('Yes', '#FF6B6B')]:
        subset = df[df['Churn'] == label][col].dropna()
        ax2.hist(subset, bins=30, alpha=0.6, color=color, label=f'Churn={label}', edgecolor='#0F1117')
    ax2.set_title(f'{col} — By Churn', color='white', fontweight='bold')
    ax2.set_xlabel(col, color='#808898')
    legend = ax2.legend(frameon=True)
    legend.get_frame().set_facecolor('#1C2130')
    for t in legend.get_texts(): t.set_color('white')

plt.suptitle('Numeric Feature Distributions', color='white', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../reports/eda_numeric_distributions.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

## 4. Contract Type vs Churn (Key Business Insight)

In [ ]:
contract_churn = df.groupby('Contract')['Churn'].apply(lambda x: (x == 'Yes').mean()).reset_index()
contract_churn.columns = ['Contract', 'ChurnRate']
contract_churn = contract_churn.sort_values('ChurnRate', ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
fig.patch.set_facecolor('#0F1117')
ax.set_facecolor('#161B27')

colors = ['#FF6B6B', '#FF9F43', '#00E5A0']
bars = ax.bar(contract_churn['Contract'], contract_churn['ChurnRate'] * 100,
              color=colors, width=0.5, zorder=2)
for bar, val in zip(bars, contract_churn['ChurnRate']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'{val:.1%}', ha='center', color='white', fontweight='bold', fontsize=12)

ax.set_ylabel('Churn Rate (%)', color='#808898')
ax.set_title('Churn Rate by Contract Type\nMonth-to-month customers churn at 5× the rate of two-year contracts',
             color='white', fontweight='bold', fontsize=12)
ax.set_axisbelow(True)
ax.yaxis.grid(True, color='#232838', zorder=0)
ax.tick_params(colors='#808898')
for spine in ax.spines.values(): spine.set_color('#232838')

plt.tight_layout()
plt.savefig('../reports/eda_contract_churn.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

## 5. Correlation Heatmap

In [ ]:
df_enc = df.copy()
df_enc['TotalCharges'] = pd.to_numeric(df_enc['TotalCharges'], errors='coerce').fillna(0)
df_enc['Churn_bin'] = (df_enc['Churn'] == 'Yes').astype(int)

numeric_df = df_enc[['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn_bin']]
corr = numeric_df.corr()

fig, ax = plt.subplots(figsize=(8, 7))
fig.patch.set_facecolor('#0F1117')
cmap = sns.diverging_palette(260, 10, s=80, l=45, as_cmap=True)
sns.heatmap(
    corr, ax=ax, cmap=cmap, center=0,
    annot=True, fmt='.3f', annot_kws={'size': 12, 'color': 'white'},
    linewidths=2, linecolor='#0F1117', square=True,
    xticklabels=corr.columns,
    yticklabels=corr.columns,
    cbar_kws={'shrink': 0.8},
)
ax.set_facecolor('#161B27')
ax.tick_params(colors='#808898', labelsize=11)
ax.set_title('Numeric Feature Correlation Matrix', color='white', fontweight='bold', fontsize=13, pad=16)
plt.tight_layout()
plt.savefig('../reports/eda_correlation.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

print('Key finding: TotalCharges has strong negative correlation with Churn (-0.20)')
print('Customers who have spent more overall are less likely to churn — loyalty effect.')